In [0]:
%sql
SELECT * FROM dev_project.flight_delay_lakehouse.bronze_flights_raw

In [0]:
from pyspark.sql.functions import from_json, to_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

#Reading bronze flights raw to a dataframe 
bronze_flights_df = spark.table("dev_project.flight_delay_lakehouse.bronze_flights_raw")

#Creating a schema for the flights raw data
schema_flights = StructType([
    StructField("flight_status", StringType(), True),
    
    StructField("departure", StructType([
        StructField("iata", StringType(), True),
        StructField("delay", IntegerType(), True),
        StructField("scheduled", StringType(), True),
        StructField("estimated", StringType(), True),
        StructField("actual", StringType(), True)
    ]), True),

    StructField("arrival", StructType([
        StructField("iata", StringType(), True),
        StructField("delay", IntegerType(), True),
        StructField("scheduled", StringType(), True),
        StructField("estimated", StringType(), True),
        StructField("actual", StringType(), True)
    ]), True),
    
    StructField("airline", StructType([
        StructField("name", StringType(), True),
        StructField("iata", StringType(), True)
    ]), True),

    StructField("flight", StructType([
        StructField("number", StringType(), True),
        StructField("iata", StringType(), True)
    ]), True)
])

#Parse flights JSON data
parsed_flights_df = bronze_flights_df.withColumn("flights_parsed", from_json(col("raw_record_json"), schema_flights))

#Creating the final flights silver table

silver_flights_df = parsed_flights_df.select(
    col("flights_parsed.flight.iata").alias("flight_iata"),
    col("flights_parsed.flight.number").alias("flight_number"),
    col("flights_parsed.airline.name").alias("airline_name"),
    col("flights_parsed.airline.iata").alias("airline_iata"),
    col("flights_parsed.flight_status").alias("flight_status"),

    col("flights_parsed.departure.iata").alias("dep_airport_iata"),
    col("flights_parsed.arrival.iata").alias("arr_airport_iata"),

    to_timestamp(col("flights_parsed.departure.scheduled")).alias("dep_scheduled_ts"),
    to_timestamp(col("flights_parsed.departure.estimated")).alias("dep_estimated_ts"),
    to_timestamp(col("flights_parsed.departure.actual")).alias("dep_actual_ts"),

    to_timestamp(col("flights_parsed.arrival.scheduled")).alias("arr_scheduled_ts"),
    to_timestamp(col("flights_parsed.arrival.estimated")).alias("arr_estimated_ts"),
    to_timestamp(col("flights_parsed.arrival.actual")).alias("arr_actual_ts"),

    col("flights_parsed.departure.delay").alias("dep_delay_minutes"),
    col("flights_parsed.arrival.delay").alias("arr_delay_minutes"),

    col("ingested_at")
)

#Writing to flights silver delta table
(
    silver_flights_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.flight_delay_lakehouse.silver_flights"))
